# ProofAgent™ — Judge-led (Colab)

`pip install proofagent-sdk` · set **`PROOFAGENT_API_KEY`** (+ optional **`OPENAI_API_KEY`** in Secrets)

Same variable names as **`examples/judge_led_quickstart.py`**: **`tested_agent_config`**, **`your_agent_handler`**, **`your_agent`**, **`byo`**, **`pa`**, **`result`**, plus **`print_full_evaluation_report`** (included in the SDK).

**Role** slug must match your project (default **`customer_support`**); see LangGraph notebook for the same pattern.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "proofagent-sdk"])

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["PROOFAGENT_API_KEY"] = userdata.get("PROOFAGENT_API_KEY")
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
except ImportError:
    pass
assert os.environ.get("PROOFAGENT_API_KEY", "").strip()

In [ ]:
import os

from proofagent import ProofAgent, TestedAgent, print_full_evaluation_report

tested_agent_config = {
    "role": "customer_support",
    "description": (
        "IT operations and support agent: triages incidents, looks up ticket status, and answers user "
        "questions with concise, policy-aware replies. Aligns with the LangGraph ReAct IT-ops example."
    ),
    "tools": [
        {"name": "ticket_stub", "description": "Look up ticket status (demo / ITSM integration)."},
    ],
}


def your_agent_handler(message: str) -> str:
    return "I can help with that. Let me check ticket status and policy."


your_agent = TestedAgent.from_json(tested_agent_config, handler=your_agent_handler)
byo = os.environ.get("OPENAI_API_KEY", "").strip()


async def main():
    async with ProofAgent.from_env(
        reasoning_provider="openai" if byo else None,
        reasoning_model="gpt-4o-mini" if byo else None,
    ) as pa:
        result = await pa.evaluate(your_agent, turns=3, verbose=True)
        print(result.label, result.score, result.run_id)
        print("\n--- Run ID ---\n", result.run_id)
        print_full_evaluation_report(result.report)


await main()